# FIN 4600 Lab: Pretty Good Privacy (PGP)

**Student notebook.** In this lab you will:

1. See the difference between *signing* and *encrypting* using two practice people, Alice and Bob.
2. Look inside your own OpenPGP key: the primary key, subkey, and self-signatures.
3. Check and certify your instructor's key using its fingerprint.
4. Send your public key to your instructor through Canvas.
5. (Session 2) Decrypt a challenge written only for you, verify who sent it, and send back a signed and encrypted answer.

**The rule to remember:** *Encrypt* with the recipient's **public** key. *Sign* with your own **private** key.

## Before you start

- Install **Gpg4win** (Windows) or **GPG Suite** (macOS). Create **one** key pair in Kleopatra or GPG Keychain using your @mtu.edu address and a strong passphrase. Mac users: if GPG Keychain offers to upload your key to a keyserver, say no.
- Run this notebook **on your own laptop** in Jupyter or VS Code, not in Google Colab. It must use the same keyring as Kleopatra or GPG Keychain.
- When GnuPG needs your passphrase, a separate passphrase window (pinentry) opens. If nothing seems to happen, look for it behind other windows.

## Setup

Run the next three cells every time you open this notebook.

In [ ]:
%pip install --quiet python-gnupg

In [ ]:
# Configuration: fill these in when your instructor gives them to you
INSTRUCTOR_KEY_FILE = 'instructor_pub.asc'                 # downloaded from Canvas
INSTRUCTOR_FPR_FROM_BOARD = 'PASTE THE FINGERPRINT FROM THE BOARD'
CHALLENGE_FILE = 'challenge_yourname.asc'                  # session 2
MY_FPR_OVERRIDE = ''   # only needed if you accidentally made more than one key pair

In [ ]:
import os, shutil, subprocess, tempfile
import gnupg

CANDIDATES = ['gpg', 'C:/Program Files/GnuPG/bin/gpg.exe', 'C:/Program Files (x86)/GnuPG/bin/gpg.exe',
              '/opt/homebrew/bin/gpg', '/usr/local/bin/gpg', '/usr/local/MacGPG2/bin/gpg']
ALGOS = {'1': 'RSA', '17': 'DSA', '18': 'ECDH', '22': 'EdDSA'}


def find_gpg():
    for c in CANDIDATES:
        path = shutil.which(c)
        if path:
            return path
    raise RuntimeError('gpg not found. Install Gpg4win (Windows) or GPG Suite (macOS), then restart Jupyter.')


def open_gpg(home=None):
    g = gnupg.GPG(gnupghome=home, gpgbinary=find_gpg())
    g.encoding = 'utf-8'
    return g


def clean(fpr):
    return (fpr or '').replace(' ', '').upper()


def show_keys(keys):
    for k in keys:
        print('PRIMARY key ', k['fingerprint'])
        print('   algorithm', ALGOS.get(k['algo'], k['algo']), '|', k['length'], 'bits | capabilities', repr(k.get('cap', '')))
        for uid in k['uids']:
            print('   user ID     ', uid)
        for sig in k.get('sigs', []):
            signer, uid, sigclass = (list(sig) + ['', '', ''])[:3]
            who = 'SELF-signature' if signer == k['keyid'] else 'certified by another key'
            print('  ', who.ljust(25), 'by', signer, 'on', repr(uid), '(class ' + str(sigclass) + ')')
        for sub in k.get('subkeys', []):
            print('   SUBKEY      ', sub[0], '| capabilities', repr(sub[1]))
        print()
    print('Capabilities: C=certify S=sign E=encrypt A=authenticate.')
    print('Signature class 13x = user-ID self-signature, 18x = subkey binding.')


def lsign(g, signer_fpr, target_fpr):
    # Locally certify target_fpr with signer_fpr (like Certify in Kleopatra, but not exportable)
    cmd = [g.gpgbinary]
    if g.gnupghome:
        cmd += ['--homedir', g.gnupghome]
    cmd += ['--batch', '--yes', '--default-key', signer_fpr, '--quick-lsign-key', target_fpr]
    subprocess.run(cmd, check=True)


gpg = open_gpg()   # YOUR real keyring, shared with Kleopatra / GPG Keychain
print('Using', gpg.gpgbinary, '| GnuPG version', '.'.join(str(v) for v in gpg.version))

---
# Part A (session 1): signing vs. encrypting in a sandbox

We create two practice people, **Alice** and **Bob**. Each gets a separate temporary keyring, just as two people have two separate laptops. These throwaway keys expire in one day, are deleted at the end, and never touch your real keyring.

In [ ]:
sandbox_homes = []


def make_person(name):
    home = tempfile.mkdtemp(prefix='pgp_' + name.lower() + '_')
    sandbox_homes.append(home)
    g = open_gpg(home)
    spec = g.gen_key_input(name_real=name, name_email=name.lower() + '@example.org',
                           key_type='RSA', key_length=2048, key_usage='sign',
                           subkey_type='RSA', subkey_length=2048, subkey_usage='encrypt',
                           expire_date='1d', no_protection=True)
    fpr = g.gen_key(spec).fingerprint
    print(name, 'created a key pair in their OWN keyring:', fpr)
    return g, fpr


alice, a_fpr = make_person('Alice')
bob, b_fpr = make_person('Bob')

### A1. Exchange public keys

Alice and Bob swap **public** keys only. Private keys never leave their owners.

In [ ]:
bob.import_keys(alice.export_keys(a_fpr))
alice.import_keys(bob.export_keys(b_fpr))
print('Bob now holds Alice public key: ', bool(bob.list_keys(keys=[a_fpr])))
print('Bob holds Alice PRIVATE key?     ', bool(bob.list_keys(keys=[a_fpr], secret=True)))

### A2. Signing: Alice uses **her private key**

Alice signs a payment instruction. Bob checks it with Alice's public key. Then someone changes the amount.

In [ ]:
msg = 'Wire $5,000 to account 12345 on Friday.'
signed = str(alice.sign(msg, keyid=a_fpr, clearsign=True))
print(signed)
print('Bob verifies with Alice public key  ->', bob.verify(signed).valid)
tampered = signed.replace('5,000', '50,000')
print('After changing $5,000 to $50,000     ->', bob.verify(tampered).valid)

**Question 1.** Look at the printed signed message. Could anyone read it? What did the signature protect, and what did it *not* protect?

*Your answer:*

### A3. Encrypting: Alice uses **Bob's public key**

In [ ]:
# always_trust skips the 'is this really Bob?' fingerprint check. Acceptable ONLY in this sandbox.
enc = str(alice.encrypt(msg, b_fpr, always_trust=True))
print(enc[:300], '...')
print()
print('Can Alice decrypt what she just sent? ->', alice.decrypt(enc).ok)
print('Can Bob decrypt it?                  ->', str(bob.decrypt(enc)))

**Question 2.** Alice wrote the message, yet she could not decrypt it. Why not? Which key would she need?

*Your answer:*

### A4. Both: sign with Alice's private key, encrypt to Bob's public key

This is exactly what your instructor does with your challenge file in session 2.

In [ ]:
both = str(alice.encrypt(msg, b_fpr, sign=a_fpr, always_trust=True))
d = bob.decrypt(both)
print('Bob reads:      ', str(d))
print('Signature valid:', d.valid, '| signed by', d.username)

### A5. Anatomy of Alice's key, as Bob sees it

Look for the **primary key**, the **user ID**, the **encryption subkey**, and the **SELF-signatures** that tie them together.

In [ ]:
show_keys(bob.list_keys(keys=[a_fpr], sigs=True))

**Question 3.** A self-signature proves that whoever holds Alice's private key attached the name *Alice*. Does it prove this key really belongs to Alice? How could Bob become sure?

*Your answer:*

### A6. Clean up the sandbox

Deletes Alice's and Bob's temporary keyrings.

In [ ]:
gpgconf = shutil.which(os.path.join(os.path.dirname(find_gpg()), 'gpgconf')) or shutil.which('gpgconf')
for home in sandbox_homes:
    if gpgconf:
        subprocess.run([gpgconf, '--homedir', home, '--kill', 'gpg-agent'], capture_output=True)
    shutil.rmtree(home, ignore_errors=True)
sandbox_homes.clear()
print('Sandbox keyrings deleted.')

---
# Part B (session 1): your own key

Create your key pair in Kleopatra or GPG Keychain first. Then run this cell.

In [ ]:
secret = {k['fingerprint'] for k in gpg.list_keys(secret=True)}
mine = [k for k in gpg.list_keys(sigs=True) if k['fingerprint'] in secret]
print('You have', len(mine), 'key pair(s). For this lab you need exactly one.')
print()
show_keys(mine)

if MY_FPR_OVERRIDE:
    MY_FPR = clean(MY_FPR_OVERRIDE)
elif len(mine) == 1:
    MY_FPR = mine[0]['fingerprint']
else:
    MY_FPR = None
    print('More than one (or zero) key pairs: set MY_FPR_OVERRIDE in the configuration cell, or delete the extra key in Kleopatra.')
print('Your fingerprint:', MY_FPR)

**Question 4.** How many subkeys does your key have, and what capability does each have? How many self-signatures do you see?

*Your answer:*

Your revocation certificate was saved automatically in the `openpgp-revocs.d` folder of your GnuPG home. Back it up somewhere safe.

---
# Part C (session 1): check and certify your instructor's key

Download `instructor_pub.asc` from Canvas into the same folder as this notebook. Copy the fingerprint your instructor writes on the board into the configuration cell, re-run that cell, then run this one. You will be asked for **your** passphrase.

In [ ]:
with open(INSTRUCTOR_KEY_FILE) as f:
    res = gpg.import_keys(f.read())
if not res.fingerprints:
    raise ValueError('No OpenPGP key found in ' + INSTRUCTOR_KEY_FILE)
INSTRUCTOR_FPR = res.fingerprints[0]
print('Key file fingerprint:  ', INSTRUCTOR_FPR)
print('Fingerprint on board:  ', clean(INSTRUCTOR_FPR_FROM_BOARD))
if INSTRUCTOR_FPR != clean(INSTRUCTOR_FPR_FROM_BOARD):
    raise ValueError('FINGERPRINT MISMATCH. Do not use this key. Tell your instructor.')
lsign(gpg, MY_FPR, INSTRUCTOR_FPR)
print('Match. Certified with your key. Kleopatra / GPG Keychain now shows it as certified.')

**Question 5.** The key file came from Canvas and the fingerprint came from the board. Why use two different channels? What could an attacker do if you only used one?

*Your answer:*

---
# Part D (session 1): send your public key to your instructor

This writes **only your public key** to a file. Your private key never leaves your laptop. Do **not** upload it to a public keyserver; we use Canvas instead.

In [ ]:
with open('my_public_key.asc', 'w') as f:
    f.write(gpg.export_keys(MY_FPR))
print('Wrote my_public_key.asc  ->  upload it to the Canvas assignment.')
print('Type this fingerprint into the Canvas quiz (a separate channel):')
print(MY_FPR)

---
# Part E (session 2): decrypt your challenge and respond

Run the **Setup** cells and the **Part B** cell again first (they set `MY_FPR`). Download your challenge file from Canvas, put its name in `CHALLENGE_FILE`, and re-run the configuration cell.

In [ ]:
INSTRUCTOR_FPR = clean(INSTRUCTOR_FPR_FROM_BOARD)
with open(CHALLENGE_FILE, 'rb') as f:
    res = gpg.decrypt_file(f)
print('Decrypted:', res.ok, '|', res.status)
print()
print(str(res))
print('----- signature check -----')
if res.valid:
    print('GOOD signature from', res.username)
    print('  signing key', res.fingerprint, '| trust:', res.trust_text)
else:
    print('NO valid signature. You cannot tell who sent this.')

**Question 6.** Which key let you read this file? Which key proved your instructor sent it? Why can't your neighbor open your file?

*Your answer:*

### E2. Write your answer, sign it, and encrypt it to your instructor

Replace the text below with your code word and your answer to the challenge question. The response is encrypted to your instructor **and** to you, so you can reread it.

In [ ]:
ANSWER = '''Code word: PASTE-YOUR-CODE-WORD
Answer: Write your answer to the challenge question here.
'''

with open('answer.txt', 'w') as f:
    f.write(ANSWER)
with open('answer.txt', 'rb') as f:
    res = gpg.encrypt_file(f, recipients=[INSTRUCTOR_FPR, MY_FPR], sign=MY_FPR,
                           armor=True, output='response.asc')
print('Wrote response.asc' if res.ok else 'FAILED: ' + res.status)
print('Upload response.asc to Canvas.')

In [ ]:
# Optional: prove to yourself it worked by decrypting your own response
with open('response.asc', 'rb') as f:
    check = gpg.decrypt_file(f)
print(str(check))
print('Signed by you:', check.valid, '|', check.username)

---
## Troubleshooting

| Message | Fix |
| --- | --- |
| gpg not found | Install Gpg4win or GPG Suite, then restart Jupyter |
| No module named gnupg, or odd errors | Run `%pip uninstall -y gnupg` then `%pip install python-gnupg` |
| More than one key pair | Set `MY_FPR_OVERRIDE`, or delete the extra key in Kleopatra |
| unusable public key / no assurance | Run the Part C cell again with the board fingerprint |
| No secret key | Your challenge was made for a different key. Tell your instructor |
| Nothing happens | Look for the passphrase window behind other windows |